<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/12-pretraining-transfer-parameter-efficient-adaptation.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Pretraining, Transfer, and Parameter-Efficient Adaptation** {#pretraining-transfer-parameter-efficient-adaptation}

Training every model from random initialization treats each task as unrelated. **Pretraining** instead learns parameters on a source objective and reuses them as an informed starting point. **Transfer learning** asks which knowledge remains useful for a target task or domain. **Adaptation** chooses what to update under constraints such as few labels, limited memory, multiple customers, or the need to retain old behavior. The central question is therefore not merely “should this model be fine-tuned?” but **where should task-specific change live, how much change is justified, and how will we detect harmful change?**

This chapter continues the representation-learning story from Chapter 11 but changes the experimental question. A classifier is first pretrained on clean handwritten digits. Its deployment input then shifts while class semantics remain unchanged: digits move one pixel, blur slightly, lose contrast, and acquire sensor noise. Only eight labeled target examples per class are available. Every method receives the same pretrained checkpoint, target split, validation rule, and evaluation metrics.

The data are scikit-learn's copy of the [UCI Optical Recognition of Handwritten Digits dataset](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B): 1,797 8×8 images, DOI [10.24432/C50P49](https://doi.org/10.24432/C50P49), CC BY 4.0. The compact setting is a **mechanism study**, not a state-of-the-art benchmark. It makes parameter updates and distribution boundaries inspectable on CPU.

![Clean source digits and their label-preserving shifted target versions.](assets/dl12-source-target-shift.svg){fig-align="center" width="76%" fig-alt="Pairs of clean and shifted blurred noisy handwritten digits for classes zero through nine."}

*Original visualization generated from the chapter's UCI-derived digit examples. The right side applies the same deterministic family of label-preserving corruptions used by the code.*

<details>
<summary><strong>PyTorch: establish the source checkpoint and low-shot target domain</strong></summary>

```python
import copy
import math
import random

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=1212):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
all_labels = torch.tensor(digits.target, dtype=torch.long)
all_indices = np.arange(len(all_images))

source_train_idx, holdout_idx = train_test_split(
    all_indices, test_size=0.30, random_state=1212, stratify=digits.target
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=1212,
    stratify=digits.target[holdout_idx],
)
source_train_x = all_images[source_train_idx]
source_train_y = all_labels[source_train_idx]
source_val_x, source_val_y = all_images[val_idx], all_labels[val_idx]
source_test_x, source_test_y = all_images[test_idx], all_labels[test_idx]


def make_target_domain(images, seed):
    # The mapping changes acquisition style, not the intended digit label.
    shifted = torch.zeros_like(images)
    shifted[:, :, :, 1:] = images[:, :, :, :-1]
    blurred = F.avg_pool2d(shifted, kernel_size=3, stride=1, padding=1)
    generator = torch.Generator().manual_seed(seed)
    noise = 0.055 * torch.randn(images.shape, generator=generator)
    return (0.82 * blurred + noise).clamp(0.0, 1.0)


target_pool_x = make_target_domain(source_train_x, seed=1213)
target_val_x = make_target_domain(source_val_x, seed=1214)
target_test_x = make_target_domain(source_test_x, seed=1215)


def balanced_low_shot_indices(labels, per_class=8, seed=1212):
    generator = torch.Generator().manual_seed(seed)
    selected = []
    for label in range(10):
        candidates = torch.where(labels == label)[0]
        selected.append(candidates[torch.randperm(len(candidates), generator=generator)[:per_class]])
    return torch.cat(selected)


target_adapt_local_idx = balanced_low_shot_indices(source_train_y, per_class=8)
target_adapt_x = target_pool_x[target_adapt_local_idx]
target_adapt_y = source_train_y[target_adapt_local_idx]


class DigitBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)

    def forward(self, images):
        flat = images.flatten(1)
        return F.relu(self.fc2(F.relu(self.fc1(flat))))


class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = DigitBackbone()
        self.head = nn.Linear(64, 10)

    def forward(self, images, return_features=False):
        features = self.backbone(images)
        logits = self.head(features)
        return (logits, features) if return_features else logits


def make_loader(images, labels, batch_size=128, shuffle=True, seed=1212):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        TensorDataset(images, labels), batch_size=batch_size, shuffle=shuffle,
        generator=generator,
    )


@torch.no_grad()
def accuracy(model, images, labels):
    model.eval()
    return float((model(images).argmax(dim=1) == labels).float().mean())


def train_with_validation(model, train_x, train_y, val_x, val_y, *, epochs=60,
                          lr=3e-3, weight_decay=1e-4, parameter_groups=None,
                          seed=1212):
    seed_everything(seed)
    trainable = [p for p in model.parameters() if p.requires_grad]
    groups = parameter_groups if parameter_groups is not None else trainable
    optimizer = torch.optim.AdamW(groups, lr=lr, weight_decay=weight_decay)
    loader = make_loader(train_x, train_y, batch_size=min(128, len(train_x)), seed=seed)
    best_state, best_val = copy.deepcopy(model.state_dict()), -1.0
    for _ in range(epochs):
        model.train()
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            loss = F.cross_entropy(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
        score = accuracy(model, val_x, val_y)
        if score > best_val:
            best_val, best_state = score, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return best_val


source_model = DigitClassifier()
train_with_validation(
    source_model, source_train_x, source_train_y, source_val_x, source_val_y,
    epochs=55, lr=3e-3, weight_decay=1e-4, seed=1212,
)
source_state = copy.deepcopy(source_model.state_dict())
source_clean_accuracy = accuracy(source_model, source_test_x, source_test_y)
zero_shot_target_accuracy = accuracy(source_model, target_test_x, source_test_y)
adaptation_results = {}

assert all_images.shape == (1797, 1, 8, 8)
assert len(set(source_train_idx) & set(test_idx)) == 0
assert torch.bincount(target_adapt_y).tolist() == [8] * 10
assert source_clean_accuracy > 0.90
assert zero_shot_target_accuracy < source_clean_accuracy
print({
    "split": (len(source_train_idx), len(val_idx), len(test_idx)),
    "target labels used": len(target_adapt_y),
    "source clean accuracy": round(source_clean_accuracy, 3),
    "zero-shot target accuracy": round(zero_shot_target_accuracy, 3),
})
```

</details>

The target validation and test sets are transformations of the original validation and test indices, never of training examples. Model selection uses target validation labels; the target test set is touched only for final reporting. The target adaptation set is drawn from source-training indices, so no transformed copy of a held-out image leaks into training.

### **Why Pretraining Transfers** {#why-pretraining-transfers}

Pretraining transfers when the source parameters encode reusable computations. Early visual layers may detect local intensity changes; deeper layers may combine them into strokes and digit parts. If the target still depends on those factors, optimizing from the source checkpoint starts inside a useful region of parameter space. Transfer can reduce labeled-data requirements, shorten optimization, and improve stability relative to random initialization.

This benefit is conditional. Let a pretrained model decompose into representation $h=f_{\theta}(x)$ and task head $\hat y=g_{\phi}(h)$. A target risk is

$$
\mathcal{R}_T(\theta,\phi)=\mathbb{E}_{(x,y)\sim P_T}\left[\ell(g_{\phi}(f_{\theta}(x)),y)\right].
$$

The source checkpoint helps only when its representation makes the target risk easier to reduce. Similar input appearance is neither necessary nor sufficient: two domains can look different but share causal features, or look similar while their labels mean different things. Negative transfer occurs when source invariances erase target information, source shortcuts remain predictive only in the old domain, or aggressive updates overfit the small target sample.

[Yosinski et al.](https://arxiv.org/abs/1411.1792) separated feature generality from co-adaptation and showed empirically that transferability decreases as tasks and upper-layer specializations diverge. The practical implication is diagnostic: compare a frozen probe, partial unfreezing, and full fine-tuning. A strong frozen probe indicates that target information is already linearly accessible; a large full-tuning gain indicates representational mismatch; both performing poorly suggests the source task, architecture, or data may be inappropriate.

Three variables dominate transfer:

- **Task relatedness:** whether the target decision depends on factors preserved by pretraining.
- **Domain shift:** whether $P_S(x)$ and $P_T(x)$ differ in ways the representation handles.
- **Target evidence:** enough labels or self-supervised signal must exist to justify changing many parameters.

The source/target accuracy gap above is therefore useful evidence. The clean checkpoint knows digit semantics, but its pixel-to-feature map is not invariant to the deployment corruption. The next methods allocate increasing adaptation capacity to close that gap.

### **Feature Extraction vs Full Fine-Tuning** {#feature-extraction-full-fine-tuning}

**Feature extraction** freezes $\theta$ and updates only the target head $\phi$. It is cheap, stable, and supports many task-specific heads over one shared backbone. Its ceiling is the information already accessible in $f_{\theta}(x)$. **Full fine-tuning** updates both $\theta$ and $\phi$, allowing the representation itself to move toward the target but increasing optimizer memory, overfitting risk, and per-task checkpoint storage.

![A spectrum from a frozen linear probe to full fine-tuning.](assets/dl12-transfer-spectrum.svg){fig-align="center" width="76%" fig-alt="A horizontal transfer spectrum showing linear probing, partial unfreezing, PEFT, and full fine-tuning from stable to plastic."}

*Original teaching diagram synthesizing the transfer strategies compared in this chapter.*

The comparison must hold initialization and target evidence constant. Both models below begin from `source_state`; neither receives extra target examples. Because the digit label space is unchanged, the pretrained head is retained and allowed to adapt. Reinitializing it would answer a different question about changing task semantics.

<details>
<summary><strong>PyTorch: compare a frozen backbone with full fine-tuning</strong></summary>

```python
def count_parameters(model, trainable_only=False):
    parameters = (p for p in model.parameters() if (p.requires_grad or not trainable_only))
    return sum(p.numel() for p in parameters)


def record_method(name, model, task_parameters=None, note=""):
    adaptation_results[name] = {
        "target_accuracy": accuracy(model, target_test_x, source_test_y),
        "source_accuracy_active": accuracy(model, source_test_x, source_test_y),
        "trainable_parameters": count_parameters(model, trainable_only=True),
        "task_parameters": task_parameters if task_parameters is not None else count_parameters(model, True),
        "note": note,
    }


feature_extractor = DigitClassifier()
feature_extractor.load_state_dict(source_state)
for parameter in feature_extractor.backbone.parameters():
    parameter.requires_grad = False
train_with_validation(
    feature_extractor, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=70, lr=8e-3, weight_decay=1e-4, seed=1220,
)
record_method("Frozen features", feature_extractor, note="head only")

full_finetune = DigitClassifier()
full_finetune.load_state_dict(source_state)
train_with_validation(
    full_finetune, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=70, lr=7e-4, weight_decay=2e-4, seed=1221,
)
record_method(
    "Full fine-tune", full_finetune,
    task_parameters=count_parameters(full_finetune), note="all weights",
)

assert adaptation_results["Frozen features"]["trainable_parameters"] == 650
assert adaptation_results["Full fine-tune"]["trainable_parameters"] > 17000
print({name: {k: round(v, 3) if isinstance(v, float) else v for k, v in result.items()}
       for name, result in adaptation_results.items()})
```

</details>

Target accuracy answers whether the adapted behavior works. Clean-source accuracy with the adaptation active answers whether the deployed task configuration damages old behavior. These are distinct from parameter retention: a frozen base can remain byte-for-byte intact while an active head or adapter changes predictions. Conversely, full fine-tuning may retain good source accuracy even though every stored weight is now task-specific.

When the target set is tiny, the validation curve is often more informative than training loss. Near-zero target training loss can coexist with poor target validation accuracy because a large model memorizes 80 examples. A lower learning rate, stronger weight decay, fewer unfrozen layers, or a PEFT method can be a better bias than “train longer.”

### **Layer Freezing and Discriminative Learning Rates** {#layer-freezing-discriminative-learning-rates}

Partial fine-tuning places a boundary inside the network. Lower layers remain stable while upper layers and the head adapt. This is useful when the target reuses basic features but needs a different composition of them. The boundary is a hypothesis about where domain-specific information lives, not a universal rule: early layers can also require adaptation under severe sensor or modality shift.

Discriminative learning rates refine the same idea. If parameter groups $G_1,\ldots,G_K$ have learning rates $\eta_1,\ldots,\eta_K$, one step is

$$
\theta_k \leftarrow \theta_k-\eta_k\nabla_{\theta_k}\mathcal{L}_T.
$$

Smaller $\eta_k$ for lower pretrained layers treats them as a strong prior; larger rates for newly initialized or task-specific layers permit faster movement. The schedule does not guarantee safety. Gradient scale, optimizer moments, normalization state, and training duration also determine the effective displacement $\|\theta_k-\theta_k^{(0)}\|$.

<details>
<summary><strong>PyTorch: unfreeze the upper block with discriminative learning rates</strong></summary>

```python
partial_tune = DigitClassifier()
partial_tune.load_state_dict(source_state)
for parameter in partial_tune.backbone.fc1.parameters():
    parameter.requires_grad = False

parameter_groups = [
    {"params": list(partial_tune.backbone.fc2.parameters()), "lr": 2e-4},
    {"params": list(partial_tune.head.parameters()), "lr": 1.5e-3},
]
train_with_validation(
    partial_tune, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=75, lr=1e-3, weight_decay=2e-4,
    parameter_groups=parameter_groups, seed=1222,
)
record_method("Partial + discriminative LR", partial_tune, note="fc2 + head")

frozen_gradients = [p.grad for p in partial_tune.backbone.fc1.parameters()]
assert all(gradient is None for gradient in frozen_gradients)
assert [group["lr"] for group in parameter_groups] == [2e-4, 1.5e-3]
print({
    "trainable parameters": count_parameters(partial_tune, True),
    "target test": round(adaptation_results["Partial + discriminative LR"]["target_accuracy"], 3),
    "clean source with adapted layers": round(adaptation_results["Partial + discriminative LR"]["source_accuracy_active"], 3),
})
```

</details>

Useful diagnostics include per-group gradient norms, parameter displacement from the checkpoint, and performance while progressively unfreezing blocks. If a newly unfrozen block receives almost no gradient, the additional capacity is not being used. If its displacement grows rapidly while validation worsens, the learning rate or target evidence is insufficiently constrained. Batch-normalization running statistics require special care: `requires_grad=False` freezes affine parameters but does not stop running-mean updates unless the module is kept in evaluation mode.

### **Domain Adaptation and Distribution Shift** {#domain-adaptation-distribution-shift}

Transfer learning is often organized by tasks, while **domain adaptation** focuses on distributions. With source distribution $P_S(x,y)$ and target distribution $P_T(x,y)$:

- **Covariate shift:** $P_S(x)\neq P_T(x)$ but $P(y\mid x)$ is assumed stable.
- **Label shift:** $P_S(y)\neq P_T(y)$ while class-conditional input distributions are treated as stable.
- **Concept shift:** $P_S(y\mid x)\neq P_T(y\mid x)$; the meaning of evidence changes.

These assumptions imply different corrections. Importance weighting can address identifiable covariate or label shift; representation alignment can reduce domain-specific variation; concept shift usually requires new labels, revised targets, or continual monitoring. Blindly aligning marginal features can be harmful when source and target class proportions differ or when classes become mixed.

![Source and target feature distributions before and after representation alignment.](assets/dl12-domain-adaptation.svg){fig-align="center" width="74%" fig-alt="Blue source circles and red target squares move from separated clusters toward overlapping feature statistics after alignment."}

*Original teaching diagram based on the feature-statistics alignment objective introduced by [Deep CORAL](https://mlanthology.org/eccv/2016/sun2016eccv-deep/).* 

CORAL aligns second-order feature statistics. For centered source features $H_S\in\mathbb{R}^{n_S\times d}$ and target features $H_T\in\mathbb{R}^{n_T\times d}$, define covariance matrices $C_S$ and $C_T$. Deep CORAL adds

$$
\mathcal{L}_{\text{CORAL}}=\frac{1}{4d^2}\|C_S-C_T\|_F^2,
\qquad
\mathcal{L}=\mathcal{L}_{\text{source-cls}}+\lambda\mathcal{L}_{\text{CORAL}}.
$$

The classifier remains anchored by labeled source examples while unlabeled target examples encourage domain-invariant features. The Frobenius norm compares all covariance entries; $4d^2$ normalizes scale. This is a mechanism demonstration, not evidence that covariance matching is universally best.

<details>
<summary><strong>PyTorch: unsupervised target alignment with Deep CORAL</strong></summary>

```python
def covariance(features):
    centered = features - features.mean(dim=0, keepdim=True)
    return centered.T @ centered / max(len(features) - 1, 1)


def coral_loss(source_features, target_features):
    dimension = source_features.shape[1]
    return (covariance(source_features) - covariance(target_features)).pow(2).sum() / (4 * dimension ** 2)


coral_model = DigitClassifier()
coral_model.load_state_dict(source_state)
optimizer = torch.optim.AdamW(coral_model.parameters(), lr=5e-4, weight_decay=2e-4)
source_alignment_x = source_train_x[:400]
source_alignment_y = source_train_y[:400]
best_state, best_val = copy.deepcopy(coral_model.state_dict()), -1.0
for _ in range(55):
    coral_model.train()
    optimizer.zero_grad()
    source_logits, source_features = coral_model(source_alignment_x, return_features=True)
    _, target_features = coral_model(target_adapt_x, return_features=True)
    classification = F.cross_entropy(source_logits, source_alignment_y)
    alignment = coral_loss(source_features, target_features)
    loss = classification + 12.0 * alignment
    loss.backward()
    optimizer.step()
    validation = accuracy(coral_model, target_val_x, source_val_y)
    if validation > best_val:
        best_val, best_state = validation, copy.deepcopy(coral_model.state_dict())
coral_model.load_state_dict(best_state)
record_method("CORAL (0 target labels)", coral_model, note="unlabeled target alignment")

with torch.no_grad():
    _, clean_features = source_model(source_alignment_x, return_features=True)
    _, shifted_features = source_model(target_adapt_x, return_features=True)
    initial_gap = float((covariance(clean_features) - covariance(shifted_features)).norm())
    _, aligned_source = coral_model(source_alignment_x, return_features=True)
    _, aligned_target = coral_model(target_adapt_x, return_features=True)
    final_gap = float((covariance(aligned_source) - covariance(aligned_target)).norm())
assert math.isfinite(final_gap)
print({
    "covariance gap before": round(initial_gap, 3),
    "covariance gap after": round(final_gap, 3),
    "target test": round(adaptation_results["CORAL (0 target labels)"]["target_accuracy"], 3),
})
```

</details>

Alignment should be monitored by class-conditional performance, calibration, and subgroup behavior, not just a smaller discrepancy. A collapsed representation can match distributions while losing task information. In production, drift detection must also ask whether the shift is inside the adaptation assumptions; a new label policy is not a sensor shift.

### **Multi-Task and Continual Learning** {#multi-task-continual-learning}

Multi-task learning optimizes related objectives together. A shared backbone receives gradients from task-specific heads:

$$
\mathcal{L}_{\text{MTL}}=\sum_{t=1}^{T}\alpha_t\mathcal{L}_t,
$$

where $\alpha_t$ controls each task's influence. Shared features can act as data-dependent regularization: digit identity and parity both require stroke evidence. But tasks can also compete. If gradients $g_i$ and $g_j$ have negative cosine similarity, $g_i^\top g_j/(\|g_i\|\|g_j\|)<0$, one task's update locally increases the other's loss. Task weighting, gradient surgery, separate normalization, or partially shared modules may be needed.

Continual learning changes the timing. Tasks or domains arrive sequentially, and old data may be unavailable. The model must balance **plasticity** for the new distribution with **stability** for earlier competence. Three broad families are:

- **Replay:** retain or generate representative old examples.
- **Regularization:** penalize changes to parameters or outputs important to old tasks.
- **Isolation:** allocate task-specific modules, masks, adapters, or experts.

The code below adds an auxiliary parity head to the same low-shot target images. This is not extra data: parity is a deterministic coarser label derived from digit identity. It demonstrates how one backbone receives two learning signals and how to inspect gradient compatibility.

<details>
<summary><strong>PyTorch: shared-backbone digit and parity learning</strong></summary>

```python
class DigitParityModel(nn.Module):
    def __init__(self, pretrained_state):
        super().__init__()
        pretrained = DigitClassifier()
        pretrained.load_state_dict(pretrained_state)
        self.backbone = copy.deepcopy(pretrained.backbone)
        self.digit_head = copy.deepcopy(pretrained.head)
        self.parity_head = nn.Linear(64, 2)

    def forward(self, images):
        features = self.backbone(images)
        return self.digit_head(features), self.parity_head(features)


multitask_model = DigitParityModel(source_state)
optimizer = torch.optim.AdamW(multitask_model.parameters(), lr=7e-4, weight_decay=2e-4)
best_state, best_digit_val = copy.deepcopy(multitask_model.state_dict()), -1.0
for _ in range(70):
    multitask_model.train()
    optimizer.zero_grad()
    digit_logits, parity_logits = multitask_model(target_adapt_x)
    digit_loss = F.cross_entropy(digit_logits, target_adapt_y)
    parity_loss = F.cross_entropy(parity_logits, target_adapt_y % 2)
    (digit_loss + 0.35 * parity_loss).backward()
    optimizer.step()
    multitask_model.eval()
    with torch.no_grad():
        val_digit, _ = multitask_model(target_val_x)
        val_score = float((val_digit.argmax(1) == source_val_y).float().mean())
    if val_score > best_digit_val:
        best_digit_val, best_state = val_score, copy.deepcopy(multitask_model.state_dict())
multitask_model.load_state_dict(best_state)
multitask_model.eval()
with torch.no_grad():
    test_digit, test_parity = multitask_model(target_test_x)
    digit_score = float((test_digit.argmax(1) == source_test_y).float().mean())
    parity_score = float((test_parity.argmax(1) == source_test_y % 2).float().mean())

# Measure task-gradient cosine at the selected checkpoint.
digit_logits, parity_logits = multitask_model(target_adapt_x)
digit_grad = torch.autograd.grad(F.cross_entropy(digit_logits, target_adapt_y),
                                 multitask_model.backbone.parameters(), retain_graph=True)
parity_grad = torch.autograd.grad(F.cross_entropy(parity_logits, target_adapt_y % 2),
                                  multitask_model.backbone.parameters())
digit_vector = torch.cat([g.flatten() for g in digit_grad])
parity_vector = torch.cat([g.flatten() for g in parity_grad])
gradient_cosine = float(F.cosine_similarity(digit_vector, parity_vector, dim=0))
assert -1.0001 <= gradient_cosine <= 1.0001
print({"digit accuracy": round(digit_score, 3), "parity accuracy": round(parity_score, 3),
       "shared-gradient cosine": round(gradient_cosine, 3)})
```

</details>

An auxiliary objective is useful only if it improves the intended frontier. A higher parity score does not compensate for worse digit recognition unless parity is itself a deployment requirement. Report every task separately, inspect task-gradient scale, and test whether gains survive removing the auxiliary head at inference.

### **Catastrophic Forgetting** {#catastrophic-forgetting}

Catastrophic forgetting is a sharp loss of earlier competence after learning new data. Gradient descent optimizes the current objective; it has no automatic obligation to preserve old functions. Even small parameter changes can cross a narrow basin or alter features used by many old examples.

[Elastic Weight Consolidation](https://pubmed.ncbi.nlm.nih.gov/28292907/) illustrates a regularization approach. It adds a quadratic penalty weighted by estimated parameter importance $F_i$:

$$
\mathcal{L}(\theta)=\mathcal{L}_{\text{new}}(\theta)
+\frac{\lambda}{2}\sum_i F_i(\theta_i-\theta_i^{*})^2.
$$

$\theta^{*}$ is the old checkpoint, and large $F_i$ discourages movement of parameters important to the old task. Replay takes a more direct route by interleaving old examples with new ones. It approximates a joint objective but introduces memory, privacy, and sampling questions.

![Conceptual source-retention trajectories for naive fine-tuning and replay-constrained adaptation.](assets/dl12-stability-plasticity.svg){fig-align="center" width="72%" fig-alt="Clean-source accuracy declines steeply during naive target updates but remains higher when replay or constraints are used."}

*Original conceptual diagram. Actual retention must be measured for the task and update stream; the curves are not experimental results.*

<details>
<summary><strong>PyTorch: mitigate source forgetting with rehearsal</strong></summary>

```python
replay_local_idx = balanced_low_shot_indices(source_train_y, per_class=8, seed=1225)
replay_x = source_train_x[replay_local_idx]
replay_y = source_train_y[replay_local_idx]

replay_model = DigitClassifier()
replay_model.load_state_dict(source_state)
optimizer = torch.optim.AdamW(replay_model.parameters(), lr=7e-4, weight_decay=2e-4)
best_state, best_joint = copy.deepcopy(replay_model.state_dict()), -1.0
for _ in range(70):
    replay_model.train()
    optimizer.zero_grad()
    target_loss = F.cross_entropy(replay_model(target_adapt_x), target_adapt_y)
    old_loss = F.cross_entropy(replay_model(replay_x), replay_y)
    (target_loss + 0.6 * old_loss).backward()
    optimizer.step()
    target_val = accuracy(replay_model, target_val_x, source_val_y)
    source_val = accuracy(replay_model, source_val_x, source_val_y)
    joint_score = target_val + 0.35 * source_val
    if joint_score > best_joint:
        best_joint, best_state = joint_score, copy.deepcopy(replay_model.state_dict())
replay_model.load_state_dict(best_state)
record_method("Full tune + replay", replay_model, note="80 stored source examples")

naive_source = adaptation_results["Full fine-tune"]["source_accuracy_active"]
replay_source = adaptation_results["Full tune + replay"]["source_accuracy_active"]
print({
    "naive target/source": (
        round(adaptation_results["Full fine-tune"]["target_accuracy"], 3), round(naive_source, 3)
    ),
    "replay target/source": (
        round(adaptation_results["Full tune + replay"]["target_accuracy"], 3), round(replay_source, 3)
    ),
})
```

</details>

Forgetting should be measured as a matrix, not one final number: evaluate each earlier task after each update. Also distinguish forgetting from ordinary distribution mismatch. If the clean checkpoint already fails on the target before any update, that gap is not forgetting. If clean performance falls after target adaptation, it is retention loss. Adapter isolation can preserve the base parameters exactly, but routing the wrong adapter can still produce behavioral failure.

### **Why Parameter Efficiency Matters** {#why-parameter-efficiency-matters}

Full fine-tuning stores and optimizes a task-specific copy of every parameter. For a model with $N$ parameters, mixed-precision Adam-style training may require a low-precision weight, gradient, FP32 master weight, and two FP32 moment buffers. The exact accounting depends on implementation, but optimizer state can dominate the checkpoint itself. If there are $K$ customers or domains, storing $K$ full models scales as $O(KN)$.

Parameter-efficient fine-tuning (PEFT) freezes the shared base and learns $m\ll N$ task parameters. It targets three different costs that should not be conflated:

- **Trainable parameters:** determine gradient and optimizer-state storage.
- **Per-task stored parameters:** determine how cheaply many variants can coexist.
- **Runtime memory and latency:** still include the frozen base and activations; fewer trainable parameters do not make the base free.

![Adapters, LoRA, and prompt methods inject task-specific capacity at different locations.](assets/dl12-peft-methods.svg){fig-align="center" width="76%" fig-alt="Three panels show an adapter bottleneck after a frozen block, low-rank matrices beside a frozen weight, and learned prompt vectors before a frozen model."}

*Original teaching diagram synthesized from [Houlsby adapters](https://proceedings.mlr.press/v97/houlsby19a.html), [LoRA](https://arxiv.org/abs/2106.09685), [prompt tuning](https://aclanthology.org/2021.emnlp-main.243/), and [prefix tuning](https://aclanthology.org/2021.acl-long.353/).* 

<details>
<summary><strong>Python: account for parameters and approximate AdamW state</strong></summary>

```python
total_parameters = count_parameters(source_model)
head_parameters = sum(p.numel() for p in source_model.head.parameters())


def approximate_training_bytes(trainable_parameters, parameter_bytes=4,
                               gradient_bytes=4, adam_moment_bytes=8):
    # This excludes activations, allocator overhead, temporary kernels, and the frozen base.
    return trainable_parameters * (parameter_bytes + gradient_bytes + adam_moment_bytes)


resource_rows = {
    "full fine-tune": (total_parameters, total_parameters),
    "head only": (head_parameters, head_parameters),
    "partial fc2 + head": (count_parameters(partial_tune, True), count_parameters(partial_tune, True)),
}
for name, (trainable, per_task) in resource_rows.items():
    print({
        "method": name,
        "trainable fraction": round(trainable / total_parameters, 4),
        "approx train-state KiB": round(approximate_training_bytes(trainable) / 1024, 1),
        "per-task FP32 KiB": round(per_task * 4 / 1024, 1),
    })

assert head_parameters == 650
assert approximate_training_bytes(total_parameters) > approximate_training_bytes(head_parameters)
```

</details>

PEFT matters most when a large base is reused across many tasks, when optimizer memory is the bottleneck, or when task modules must be swapped independently. It matters less when the model is already tiny, the target domain is radically different, or deployment requires merging all variants into separate binaries anyway. Parameter count is a constraint, not the objective; target quality, retention, calibration, and operational simplicity remain primary.

### **Adapter-Based Fine-Tuning** {#adapter-based-fine-tuning}

An adapter inserts a small residual bottleneck into a frozen network. For hidden state $h\in\mathbb{R}^{d}$ and bottleneck width $b\ll d$,

$$
\operatorname{Adapter}(h)=h+W_{\text{up}}\,\sigma(W_{\text{down}}h),
$$

where $W_{\text{down}}\in\mathbb{R}^{b\times d}$ and $W_{\text{up}}\in\mathbb{R}^{d\times b}$. Initializing the up projection near zero makes the module begin close to an identity residual, so adaptation does not immediately overwrite the pretrained function. [Houlsby et al.](https://proceedings.mlr.press/v97/houlsby19a.html) demonstrated this design for Transformer transfer, but the mechanism applies to any hidden representation.

Adapters provide explicit task modularity: keep one immutable base and load a small module per domain. Their costs include extra sequential operations, adapter-placement choices, and possible serving complexity when many adapters must be batched. Bottleneck width trades capacity against storage and latency.

<details>
<summary><strong>PyTorch: insert and train a residual bottleneck adapter</strong></summary>

```python
class AdapterClassifier(nn.Module):
    def __init__(self, pretrained_state, bottleneck=8):
        super().__init__()
        pretrained = DigitClassifier()
        pretrained.load_state_dict(pretrained_state)
        self.backbone = copy.deepcopy(pretrained.backbone)
        self.head = copy.deepcopy(pretrained.head)
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False
        self.adapter = nn.Sequential(
            nn.Linear(64, bottleneck), nn.ReLU(), nn.Linear(bottleneck, 64)
        )
        nn.init.zeros_(self.adapter[-1].weight)
        nn.init.zeros_(self.adapter[-1].bias)

    def forward(self, images):
        features = self.backbone(images)
        adapted = features + self.adapter(features)
        return self.head(adapted)


adapter_model = AdapterClassifier(source_state, bottleneck=8)
train_with_validation(
    adapter_model, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=75, lr=2e-3, weight_decay=2e-4, seed=1230,
)
adapter_task_parameters = sum(p.numel() for p in adapter_model.adapter.parameters()) + sum(
    p.numel() for p in adapter_model.head.parameters()
)
record_method("Adapter", adapter_model, task_parameters=adapter_task_parameters, note="bottleneck=8 + head")

assert all(not p.requires_grad for p in adapter_model.backbone.parameters())
assert adapter_task_parameters < total_parameters
print({
    "adapter + head parameters": adapter_task_parameters,
    "fraction of full model": round(adapter_task_parameters / total_parameters, 4),
    "target test": round(adaptation_results["Adapter"]["target_accuracy"], 3),
})
```

</details>

Debugging starts with identity behavior: before training, a zero-initialized adapter should reproduce the base output up to numerical equality. During training, verify that base gradients remain `None`, adapter norms leave zero, and validation gain is not caused by a changed evaluation mode. If performance saturates, widen the bottleneck or place adapters at more layers before assuming full fine-tuning is necessary.

### **LoRA** {#lora}

Low-Rank Adaptation (LoRA) represents a task-specific weight update without storing a dense matrix. For frozen $W_0\in\mathbb{R}^{d_{out}\times d_{in}}$,

$$
y=W_0x+\Delta Wx,
\qquad
\Delta W=\frac{\alpha}{r}BA,
$$

where $A\in\mathbb{R}^{r\times d_{in}}$, $B\in\mathbb{R}^{d_{out}\times r}$, and rank $r\ll\min(d_{in},d_{out})$. The trainable count changes from $d_{out}d_{in}$ to $r(d_{in}+d_{out})$. The scaling $\alpha/r$ separates update magnitude from rank. Initializing $B=0$ makes $\Delta W=0$ initially while random $A$ permits gradients to enter $B$.

The LoRA hypothesis is not that pretrained weights are low rank. It is that the **task-specific displacement** can often be represented in a low-dimensional subspace. [Hu et al.](https://arxiv.org/abs/2106.09685) introduced LoRA for large language models and emphasized that the learned update can be merged into $W_0$ for inference, avoiding an adapter's extra sequential layer.

<details>
<summary><strong>PyTorch: implement LoRA for the digit backbone</strong></summary>

```python
class LoRALinear(nn.Module):
    def __init__(self, base_layer, rank=4, alpha=8.0):
        super().__init__()
        self.base = copy.deepcopy(base_layer)
        for parameter in self.base.parameters():
            parameter.requires_grad = False
        self.rank = rank
        self.scale = alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base_layer.in_features))
        self.B = nn.Parameter(torch.zeros(base_layer.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, inputs):
        low_rank = F.linear(F.linear(inputs, self.A), self.B)
        return self.base(inputs) + self.scale * low_rank

    def merged_weight(self):
        return self.base.weight + self.scale * (self.B @ self.A)


class LoRAClassifier(nn.Module):
    def __init__(self, pretrained_state, rank=4):
        super().__init__()
        pretrained = DigitClassifier()
        pretrained.load_state_dict(pretrained_state)
        self.fc1 = LoRALinear(pretrained.backbone.fc1, rank=rank)
        self.fc2 = LoRALinear(pretrained.backbone.fc2, rank=rank)
        self.head = copy.deepcopy(pretrained.head)

    def forward(self, images):
        flat = images.flatten(1)
        features = F.relu(self.fc2(F.relu(self.fc1(flat))))
        return self.head(features)


lora_model = LoRAClassifier(source_state, rank=4)
with torch.no_grad():
    base_difference = float((lora_model(target_adapt_x) - source_model(target_adapt_x)).abs().max())
train_with_validation(
    lora_model, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=75, lr=1.8e-3, weight_decay=2e-4, seed=1231,
)
lora_task_parameters = count_parameters(lora_model, True)
record_method("LoRA rank 4", lora_model, task_parameters=lora_task_parameters, note="two LoRA layers + head")

assert base_difference < 1e-6
assert lora_task_parameters < total_parameters
assert lora_model.fc1.merged_weight().shape == lora_model.fc1.base.weight.shape
print({
    "initial max output difference": base_difference,
    "trainable parameters": lora_task_parameters,
    "target test": round(adaptation_results["LoRA rank 4"]["target_accuracy"], 3),
})
```

</details>

Rank is a capacity hyperparameter, not a quality guarantee. Too small a rank underfits the required displacement; too large a rank erodes the storage advantage and can overfit. Layer selection can matter more than a uniform rank. Monitor $\|BA\|$, compare ranks under a fixed parameter budget, and verify merge equivalence numerically before deployment. Merging removes task-swapping flexibility unless the base or deltas are retained separately.

### **QLoRA** {#qlora}

QLoRA reduces the memory of the **frozen base** while training LoRA parameters. In the full method of [Dettmers et al.](https://proceedings.neurips.cc/paper_files/paper/2023/hash/1feb87871436031bdc0f2beaa62a049b-Abstract-Conference.html), base weights use 4-bit NormalFloat (NF4), quantization constants are themselves compressed through double quantization, and paged optimizers manage memory spikes. Computation dequantizes base blocks to a suitable compute dtype; gradients flow through those operations into LoRA parameters, while the quantized base remains frozen.

![LoRA retains a floating-point frozen base, while QLoRA stores the base in 4-bit form and learns the same low-rank path.](assets/dl12-lora-qlora.svg){fig-align="center" width="76%" fig-alt="Two panels compare a floating-point frozen base plus LoRA matrices with a four-bit frozen base that is dequantized for compute plus LoRA matrices."}

*Original teaching diagram based on the [LoRA paper](https://arxiv.org/abs/2106.09685) and the [QLoRA NeurIPS paper](https://proceedings.neurips.cc/paper_files/paper/2023/hash/1feb87871436031bdc0f2beaa62a049b-Abstract-Conference.html).* 

The runnable example below is intentionally narrower: it uses **symmetric per-output-row int4 quantization** to expose the data path. It is not NF4, does not implement double quantization or paged optimizers, and should not be called a production QLoRA implementation. That distinction matters because “4-bit weights plus LoRA” omits several techniques responsible for QLoRA's memory and quality behavior.

<details>
<summary><strong>PyTorch: demonstrate a quantized frozen base with a LoRA path</strong></summary>

```python
def symmetric_int4_quantize(weight):
    scale = weight.abs().amax(dim=1, keepdim=True).clamp_min(1e-8) / 7.0
    quantized = torch.round(weight / scale).clamp(-7, 7).to(torch.int8)
    return quantized, scale


class QuantizedLoRALinear(nn.Module):
    def __init__(self, base_layer, rank=4, alpha=8.0):
        super().__init__()
        quantized, scale = symmetric_int4_quantize(base_layer.weight.detach())
        self.register_buffer("qweight", quantized)
        self.register_buffer("weight_scale", scale)
        self.register_buffer("base_bias", base_layer.bias.detach().clone())
        self.rank = rank
        self.lora_scale = alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base_layer.in_features))
        self.B = nn.Parameter(torch.zeros(base_layer.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, inputs):
        dequantized = self.qweight.float() * self.weight_scale
        base_output = F.linear(inputs, dequantized, self.base_bias)
        return base_output + self.lora_scale * F.linear(F.linear(inputs, self.A), self.B)


class ToyQLoRAClassifier(nn.Module):
    def __init__(self, pretrained_state, rank=4):
        super().__init__()
        pretrained = DigitClassifier()
        pretrained.load_state_dict(pretrained_state)
        self.fc1 = QuantizedLoRALinear(pretrained.backbone.fc1, rank=rank)
        self.fc2 = QuantizedLoRALinear(pretrained.backbone.fc2, rank=rank)
        self.head = copy.deepcopy(pretrained.head)

    def forward(self, images):
        flat = images.flatten(1)
        features = F.relu(self.fc2(F.relu(self.fc1(flat))))
        return self.head(features)


qlora_demo = ToyQLoRAClassifier(source_state, rank=4)
dequantized_fc1 = qlora_demo.fc1.qweight.float() * qlora_demo.fc1.weight_scale
reference_fc1 = source_model.backbone.fc1.weight.detach()
relative_error = float((dequantized_fc1 - reference_fc1).norm() / reference_fc1.norm())
train_with_validation(
    qlora_demo, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=75, lr=1.8e-3, weight_decay=2e-4, seed=1232,
)
qlora_task_parameters = count_parameters(qlora_demo, True)
record_method("Toy int4 + LoRA", qlora_demo, task_parameters=qlora_task_parameters,
              note="symmetric int4 demo, not NF4 QLoRA")

assert qlora_demo.fc1.qweight.dtype == torch.int8
assert qlora_demo.fc1.qweight.requires_grad is False
assert qlora_task_parameters == lora_task_parameters
print({
    "fc1 relative quantization error": round(relative_error, 4),
    "trainable parameters": qlora_task_parameters,
    "target test": round(adaptation_results["Toy int4 + LoRA"]["target_accuracy"], 3),
})
```

</details>

Quantization adds a second approximation axis. Diagnose the frozen quantized checkpoint before adaptation, compare quantized and unquantized logits, and identify sensitive layers. QLoRA reduces base-weight storage and optimizer pressure, but activations, LoRA states, dequantization buffers, sequence length, and kernel support still determine whether the run fits in memory.

### **Prompt Tuning and Prefix Tuning** {#prompt-tuning-prefix-tuning}

Prompt methods adapt a frozen model by learning continuous inputs rather than weight deltas. **Prompt tuning** prepends or injects trainable embeddings $P\in\mathbb{R}^{m\times d}$ at the input. For a token sequence $X$, the model receives $[P;X]$. [Lester et al.](https://aclanthology.org/2021.emnlp-main.243/) found that prompt tuning becomes more competitive with full tuning as language-model scale grows.

**Prefix tuning** injects learned key/value-like states into multiple Transformer layers. If a layer would attend to $K,V$, it instead uses

$$
K'=[P_K;K],\qquad V'=[P_V;V].
$$

The prefix can influence attention at every layer without changing the base projections. It is more invasive than an input-only prompt and adds attention length, which increases key/value memory and compute. [Li and Liang](https://aclanthology.org/2021.acl-long.353/) introduced the method for conditional generation.

The digit MLP has no token sequence or attention cache, so the appropriate analogue is a **visual prompt**: a learned 8×8 additive pattern applied to every target image while the classifier remains frozen. The example demonstrates prompt-only adaptation, not Transformer prefix tuning. [Visual Prompt Tuning](https://www.ecva.net/papers/eccv_2022/papers_ECCV/html/4175_ECCV_2022_paper.php) provides the canonical vision-side context.

<details>
<summary><strong>PyTorch: learn a visual prompt while freezing the classifier</strong></summary>

```python
class VisualPromptClassifier(nn.Module):
    def __init__(self, pretrained_state):
        super().__init__()
        self.base = DigitClassifier()
        self.base.load_state_dict(pretrained_state)
        for parameter in self.base.parameters():
            parameter.requires_grad = False
        self.prompt = nn.Parameter(torch.zeros(1, 1, 8, 8))

    def forward(self, images):
        prompted = (images + 0.35 * torch.tanh(self.prompt)).clamp(0.0, 1.0)
        return self.base(prompted)


prompt_model = VisualPromptClassifier(source_state)
train_with_validation(
    prompt_model, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=100, lr=2e-2, weight_decay=1e-4, seed=1233,
)
prompt_parameters = prompt_model.prompt.numel()
record_method("Visual prompt", prompt_model, task_parameters=prompt_parameters,
              note="64-pixel additive prompt")

assert prompt_parameters == 64
assert all(not p.requires_grad for p in prompt_model.base.parameters())
assert float(prompt_model.prompt.detach().abs().max()) > 0
print({
    "prompt parameters": prompt_parameters,
    "target test": round(adaptation_results["Visual prompt"]["target_accuracy"], 3),
    "clean source with prompt active": round(adaptation_results["Visual prompt"]["source_accuracy_active"], 3),
})
```

</details>

Prompt methods can be extraordinarily compact, but they depend strongly on model scale, prompt length, initialization, and whether pretraining made the model controllable through its input interface. They may also consume context positions or add key/value cache. A prompt that works only with a particular tokenizer, template, or image normalization is operationally coupled to that preprocessing contract.

### **Full Fine-Tuning vs Parameter-Efficient Adaptation** {#full-fine-tuning-parameter-efficient-adaptation}

There is no universally dominant adaptation method. Full fine-tuning maximizes direct capacity but duplicates the model and can forget. Frozen features are stable and cheap but cannot repair the representation. Adapters create modular residual computation. LoRA creates mergeable low-rank weight updates. QLoRA additionally compresses the frozen base during training. Prompt methods move task capacity into learned inputs but may consume context or depend on scale.

The table below is generated from the shared experiment. Accuracy differences on 270 test images are noisy and seed-dependent; they illustrate mechanisms, not a leaderboard. `source_accuracy_active` evaluates the clean source while the task-specific module remains active. `task_parameters` counts what must be stored per task in this educational implementation and excludes the one shared base.

<details>
<summary><strong>Python: consolidate quality, retention, and storage evidence</strong></summary>

```python
ordered_methods = [
    "Frozen features", "Partial + discriminative LR", "Full fine-tune",
    "Full tune + replay", "CORAL (0 target labels)", "Adapter",
    "LoRA rank 4", "Toy int4 + LoRA", "Visual prompt",
]
comparison_rows = []
for method in ordered_methods:
    result = adaptation_results[method]
    comparison_rows.append({
        "method": method,
        "target_acc": round(result["target_accuracy"], 3),
        "clean_acc_active": round(result["source_accuracy_active"], 3),
        "trainable": result["trainable_parameters"],
        "per_task_KiB_FP32": round(result["task_parameters"] * 4 / 1024, 2),
        "note": result["note"],
    })

header = f'{"method":<30} {"target":>7} {"clean":>7} {"trainable":>10} {"task KiB":>9}'
print(header)
print("-" * len(header))
for row in comparison_rows:
    print(f'{row["method"]:<30} {row["target_acc"]:>7.3f} {row["clean_acc_active"]:>7.3f} '
          f'{row["trainable"]:>10d} {row["per_task_KiB_FP32"]:>9.2f}')

assert len(comparison_rows) == 9
assert next(r for r in comparison_rows if r["method"] == "Visual prompt")["trainable"] == 64
assert next(r for r in comparison_rows if r["method"] == "Full fine-tune")["per_task_KiB_FP32"] > next(
    r for r in comparison_rows if r["method"] == "LoRA rank 4"
)["per_task_KiB_FP32"]
```

</details>

| Strategy | Representation can move? | Per-task artifact | Inference implication | Characteristic risk |
|---|---:|---|---|---|
| Frozen features | No | Head | Shared backbone plus selected head | Target information not linearly accessible |
| Partial tuning | Upper layers | Updated layers and head | Usually a task-specific checkpoint | Wrong freeze boundary |
| Full fine-tuning | Yes, all layers | Full model | No extra module | Overfitting, forgetting, storage |
| Adapter | Through residual modules | Adapter and head | Extra sequential operations | Bottleneck/placement underfit |
| LoRA | Through low-rank weight deltas | $A,B$ and head | Mergeable or swappable | Insufficient rank/layer coverage |
| QLoRA | Same LoRA delta; base quantized | LoRA plus quantization metadata | Depends on merge/quantized kernels | Quantization sensitivity |
| Prompt/prefix | Base frozen; inputs/states move | Learned prompt vectors | Extra tokens, K/V, or preprocessing | Weak controllability, context cost |

A practical selection sequence is:

1. Establish zero-shot and frozen-feature baselines.
2. If representation mismatch remains, try partial tuning or a modest PEFT module.
3. Compare under the same labels, validation budget, and checkpoint rule.
4. Measure target quality, source retention, calibration, train memory, task storage, and serving latency.
5. Escalate to full fine-tuning only when its quality gain justifies the operational cost.

Method choice is deployment architecture. A merged LoRA may be ideal for one static model; swappable adapters may suit many tenants; prompt tuning may be convenient when only an embedding interface is available; full tuning may be justified for a single high-value domain with abundant target evidence.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Pretraining provides a reusable starting point; transfer is the empirical claim that this starting point lowers target risk; adaptation controls how much of the function changes. The chapter's progression forms a capacity ladder:

- **Frozen features** ask whether target information is already exposed.
- **Partial and full tuning** progressively relax the source prior.
- **Domain alignment** uses distribution structure, sometimes without target labels.
- **Replay or regularization** protects earlier competence during sequential updates.
- **Adapters, LoRA, QLoRA, and prompt methods** localize task-specific capacity to compact artifacts.

The most important distinctions are easy to lose in parameter-count comparisons. Frozen parameters still occupy inference memory. Preserving base bytes does not guarantee preserved behavior when a task module is active. QLoRA is not generic int4 quantization plus a low-rank layer. Prompt tuning and prefix tuning inject learned state at different locations. Domain adaptation is only valid under explicit shift assumptions.

For a new project, retain four pieces of evidence: a no-adaptation baseline, a full-tuning upper reference, at least one parameter-efficient alternative, and a retention evaluation on important source behavior. Track where labels enter, how checkpoints are selected, and what each task must store. This turns adaptation from a fashionable method name into an auditable decision about **capacity, evidence, stability, and systems cost**.

Chapter 13 expands the scale of this problem. Foundation models reuse one pretrained base across many tasks and modalities, making data curation, scaling behavior, model routing, and lifecycle governance as important as the local adaptation rule studied here.